# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [ ]:
# imports
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [ ]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [ ]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = MODEL_GPT
openai = OpenAI()

In [ ]:
site_url = "https://ourworldindata.org/grapher/annual-number-of-deaths-by-cause?country=~WHO_AMR"

In [ ]:
links = fetch_website_links(site_url)
links

In [ ]:
len(links)

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a conscise summary of accidental death causes (not disease related or natural causes),
Do not include Terms of Service, Privacy, email links
Do not include links that are due to disease, old age.
Do not include ones that are not a specific category, a loosely defined category is fine, but not something like 'causes of death' without even categorising the cause
It's EXTREMELY important that we don't miss any deaths that could be considered accidental from this list
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these web links have high relevance for the topic of investigating the main causes of accidental death (not disease related or natural causes), 
respond with the full https URL in JSON format.

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
   # print(f"{link_system_prompt}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
def fetch_page_and_all_relevant_links(url):
 #   contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n\n## Relevant Links:\n"  #{contents}
    for link in relevant_links['links']:
      print(link)
   #   result += f"\n\n### Link: {link['type']}\n"
      result += fetch_website_contents(link)
    return result

In [ ]:
print(fetch_page_and_all_relevant_links(site_url))

In [ ]:

summary_system_prompt = """
 You are an assistant that analyzes the contents of several relevant pages from a health website
 and creates a short, consice summary about accidental deaths around the globe. being sure to take into account prevenlance, focussing more on the bigger hitters
 Respond in markdown without code blocks.
 Include details of rates, locations, simiarities across locations and the globe in general. We are looking for the output of this to provide a useful steer for future analysis
 """


In [ ]:
def get_summary_user_prompt( url):
    user_prompt = f"""
Here are the contents of its landing page and other relevant pages;
use this information to build a short summary of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_summary_user_prompt(site_url)

In [ ]:
def create_summary(url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": summary_system_prompt},
            {"role": "user", "content": get_summary_user_prompt( url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_summary( site_url )